In [1]:
from stable_baselines3 import PPO
import numpy as np
import pandas as pd
from typing import Any
from OPSDesignerEnv import OPSDesignerEnv

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\deepchem\feat\smiles_tokenizer.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
fragment_pool = pd.read_csv("source/SMILES_list_100.csv")["SMILES"].tolist()
targetOPS = [-5.9155, -2.7331, 2.5561, 0.0639, 2.4273, 0.1288]

eval_env = OPSDesignerEnv(smiles=fragment_pool,
                          target_ops=targetOPS,
                          min_k=3,
                          max_k=3,
                          max_steps=4,
                          eps=1e-8,
                          r_cap=100,
                         )

def generate_SMILES(model, env, n_episodes=200, base_seed=42, deterministic=False):
    rows = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=base_seed + ep)
        terminated = truncated = False
        total_reward = 0.0
        last_info = {}

        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, terminated, truncated, info = env.step(int(action))
            total_reward += float(r)
            last_info = info

        best = last_info.get("best_smiles")
        selected = last_info.get("selected_smiles")
        smiles = best if best is not None else selected

        if isinstance(smiles, list):
            smiles = tuple(smiles)
        if isinstance(selected, list):
            selected = tuple(selected)

        rows.append({"episode": ep, "smiles": smiles, "selected_fragments": selected, "reward": total_reward})
        #print(f"episode {ep} done.")
        
    return pd.DataFrame(rows)

In [3]:
model = PPO.load(f"RLmodels/ppo_rm.zip", device="cpu")
df = generate_SMILES(model, eval_env, n_episodes=50)
df.to_csv(f"generated_OPSs/rl_candidates.csv", index=False)
df

,episode,smiles,selected_fragments,reward
0,0,N#Cc1sc(-c2ccc3c(c2)C(=O)NC3=O)cc1-c1ccc2c(c1)...,"(O=C1c2ccccc2-c2ccccc21, N#Cc1cccs1, O=C1NC(=O...",6.727488
1,1,Nc1ccc(-c2ccc(N)c(N3C(=O)c4ccccc4C3=O)c2)cc1,"(Nc1ccccc1, Nc1ccccc1, O=C1NC(=O)c2ccccc21)",9.442572
2,2,Nc1ccc(-c2ccc3c(c2)C(=O)c2cc(-c4ccc5c(c4)C(=O)...,"(Nc1ccccc1, O=C1c2ccccc2-c2ccccc21, O=C1c2cccc...",5.533801
3,3,N#Cc1sc(N2C(=O)c3ccccc3C2=O)cc1N1C(=O)c2ccccc2...,"(N#Cc1cccs1, O=C1NC(=O)c2ccccc21, O=C1NC(=O)c2...",8.758707
4,4,O=C1c2ccccc2C(=O)N1Cn1ccc2cc(N3C(=O)c4ccccc4C3...,"(O=C1NC(=O)c2ccccc21, Cn1ccc2ccccc21, O=C1NC(=...",8.080849
5,5,Cn1cc(N2C(=O)c3cccc(N4C(=O)c5ccccc5C4=O)c3C2=O...,"(Cn1ccc2ccccc21, O=C1NC(=O)c2ccccc21, O=C1NC(=...",11.106863
6,6,O=C1c2ccccc2-c2ccc(N3C(=O)c4ccc(-n5c6ccccc6c6c...,"(c1ccc2c(c1)Nc1ccccc12, O=C1NC(=O)c2ccccc21, O...",22.048116
7,7,Cn1cc(-c2ccc3c(c2)C(=O)N(c2cccc4c2-c2ccccc2C4=...,"(O=C1c2ccccc2-c2ccccc21, O=C1NC(=O)c2ccccc21, ...",25.393179
8,8,O=C1c2ccc(N3c4ccccc4CCc4ccccc43)cc2C(=O)N1c1cc...,"(c1ccc2c(c1)CCc1ccccc1N2, O=C1NC(=O)c2ccccc21,...",7.446054
9,9,O=C1NC(=O)c2cc(N3C(=O)c4cccc(N5C(=O)c6ccccc6C5...,"(O=C1NC(=O)c2ccccc21, O=C1NC(=O)c2ccccc21, O=C...",2.948665
